# Notebook 14 — Relevance keyword analysis

Mine training data (newsletter items 1-104, 1109 curator-approved articles)
for distinctive education terms. Output: a hand-reviewable keyword list to use
as a per-source `require_keywords` filter in `rss_adapter.py` for broad sources
(Belfast Telegraph whole-paper feed, BBC general, IfG, JRF, etc.).

**Re-run when:**
- Training data (`train.csv` / `val.csv`) is updated with newer newsletters
- New rejected articles in `data/archive/rejected/` look like false negatives → add their keywords
- A new broad source is added and we want to re-validate keyword coverage

**Built artifacts:**
- `src/scraping/relevance.py` — `DEFAULT_EDUCATION_KEYWORDS` constant and helpers
- `src/scraping/rss_adapter.py` — accepts `apply_relevance_filter=True` param
- Per-source flag in `src/scraping/sources.yml`

## Cell 1 — Setup

In [ ]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "modelling"

print("Notebook 14 — relevance keyword analysis")

## Cell 2 — Load curator-approved corpus (train + val)

In [ ]:
train = pd.read_csv(DATA / "train.csv")
val = pd.read_csv(DATA / "val.csv")
nl = pd.concat([train, val], ignore_index=True)

print(f"Combined corpus: {len(nl)} curator-published items")
print(f"  Date range: {nl['issue_date'].min()} -> {nl['issue_date'].max()}")
print(f"\nSample 'text' field:")
print(nl["text"].iloc[0][:300])

## Cell 3 — Tokenize, count unigrams + bigrams

In [ ]:
TOKEN_RE = re.compile(r"[a-z][a-z'-]+[a-z]")

STOPWORDS = set('''
a about above after again against all am an and any are as at be because been before
being below between both but by can cannot could did do does doing don down during
each few for from further had has have having he her here hers herself him himself
his how i if in into is it its itself just like me more most my myself no nor not
now of off on once only or other our ours ourselves out over own same she should
so some such than that the their theirs them themselves then there these they this
those through to too under until up very was we were what when where which while
who whom why will with would you your yours yourself yourselves
also one new use using used would could may say says said also however been per get
year years week last next time make made take taken get got see seen go going gone
say says said way ways need needs needed find found set sets setting still well good
much many lot lots well even must might shall let going us off via through into among
across down up around back forth without within across despite although though since
yet thus hence therefore moreover furthermore yes no maybe perhaps probably possibly
think thought know knew look looked tell told ask asked want wanted try tried like
liked feel felt seem seemed appear appeared become became come came give gave got
keep kept put bring brought show showed made make help helped work worked called
called help helped help help one two three four five six seven eight nine ten today
'''.split())

def tokenize(text):
    if not isinstance(text, str): return []
    return [t for t in TOKEN_RE.findall(text.lower()) if t not in STOPWORDS]

def bigrams(tokens):
    return [f"{a} {b}" for a, b in zip(tokens, tokens[1:])]

unigram_counts = Counter()
bigram_counts = Counter()
for txt in nl["text"].dropna():
    toks = tokenize(txt)
    unigram_counts.update(toks)
    bigram_counts.update(bigrams(toks))

print(f"Total tokens: {sum(unigram_counts.values()):,}")
print(f"Unique unigrams: {len(unigram_counts):,}")
print(f"Unique bigrams: {len(bigram_counts):,}")

print(f"\nTop 60 unigrams:")
for term, c in unigram_counts.most_common(60):
    print(f"  {term:<30} {c:>5}")

print(f"\nTop 40 bigrams:")
for bg, c in bigram_counts.most_common(40):
    print(f"  {bg:<40} {c:>5}")

## Cell 4 — Candidate keyword list

The list below was hand-curated from the unigram + bigram analysis above.
Edit it here in the notebook, re-run cells below to test, then **copy the final
list into `src/scraping/relevance.py`** as `DEFAULT_EDUCATION_KEYWORDS`.

In [ ]:
KEYWORDS = [
    # Core schools + sectors
    "school", "schools", "pupil", "pupils", "student", "students",
    "teacher", "teachers", "teaching", "classroom",
    "primary", "secondary", "nursery", "early years", "eyfs",
    "academy", "academies", "trust", "trusts",
    "college", "colleges", "further education", "sixth form",
    "university", "universities", "higher education", "campus",
    "education", "educational",
    # Curriculum & assessment
    "curriculum", "gcse", "a-level", "a level", "phonics",
    "literacy", "numeracy", "stem",
    "ofsted", "ofqual", "inspection",
    "exam", "exams", "examination",
    # SEND & inclusion (UK + Scottish terms)
    "send", "ehcp", "special needs", "special educational",
    "additional support for learning", "asl", "additional support needs", "asn",
    "alternative provision",
    "disability", "disabilities",
    "free school meals", "fsm", "disadvantage", "disadvantaged",
    "attendance",
    "safeguarding", "online safety", "behaviour", "exclusion", "exclusions",
    # Workforce
    "recruitment", "retention", "training", "cpd",
    "headteacher", "head teacher",
    "professional learning",
    "early career framework", "ecf",
    "governance", "trustees",
    # Policy bodies / actors
    "dfe", "department for education", "department education",
    "schools week", "schoolsweek",
    "select committee", "education committee",
    "minister", "education minister", "education secretary",
    # Pedagogy / research
    "research", "evidence-based", "pedagogy",
    "education research", "education policy",
    # Demographic / broader
    "child", "children", "young people", "youth",
    "skills", "apprenticeship", "apprenticeships",
    # Specific frequently-newslettered topics
    "white paper", "schools bill", "curriculum assessment",
    "child poverty", "mental health", "wellbeing",
]

def compile_patterns(kws):
    pats = []
    for kw in kws:
        kw_l = kw.lower().strip()
        if " " in kw_l or "-" in kw_l:
            pats.append((kw_l, re.compile(rf"(?<!\w){re.escape(kw_l)}(?!\w)")))
        else:
            pats.append((kw_l, re.compile(rf"\b{re.escape(kw_l)}\b")))
    return pats

patterns = compile_patterns(KEYWORDS)

def matches(text):
    if not isinstance(text, str): return False
    t = text.lower()
    return any(p.search(t) for _, p in patterns)

print(f"Candidate keyword list: {len(KEYWORDS)} keywords")

## Cell 5 — Test the filter on the 854 backfill articles

In [ ]:
clf = pd.read_csv(DATA / "classified_articles.csv")
clf["_haystack"] = (clf["title"].fillna("") + " " + clf["text_clean"].fillna("")).str.lower()
clf["passes"] = clf["_haystack"].apply(matches)

print(f"Filter test on {len(clf)} articles:")
print(f"  Keep:   {clf['passes'].sum()} ({clf['passes'].mean()*100:.1f}%)")
print(f"  Reject: {(~clf['passes']).sum()} ({(~clf['passes']).mean()*100:.1f}%)")

print(f"\nPer-source pass rate (sources with >=5 articles, sorted lowest-first):")
by_src = clf.groupby("source").agg(n=("url","size"), n_pass=("passes","sum"))
by_src["pct_pass"] = (by_src["n_pass"] / by_src["n"] * 100).round(1)
by_src["n_reject"] = by_src["n"] - by_src["n_pass"]
by_src = by_src[by_src["n"] >= 5].sort_values("pct_pass")
print(by_src.to_string())

## Cell 6 — Sanity check: all curator-published articles must pass

If any curator-approved article gets rejected by the filter, the keyword list is too narrow.

In [ ]:
def norm_url(u):
    if not isinstance(u, str): return None
    u = u.strip().lower()
    u = re.sub(r"^https?://(www\.)?", "", u).rstrip("/")
    u = re.sub(r"[?#].*$", "", u)
    return u or None

nl_urls = set(filter(None, (norm_url(u) for u in nl["link"].dropna())))
clf["_norm_url"] = clf["url"].apply(norm_url)
clf["in_nl"] = clf["_norm_url"].isin(nl_urls)
pub = clf[clf["in_nl"]]

print(f"Sanity check — {len(pub)} curator-published articles in backfill:")
print(f"  Pass filter: {pub['passes'].sum()} / {len(pub)}")
if (~pub["passes"]).any():
    print(f"\n⚠ FALSE NEGATIVES (published but filtered out — keywords too narrow):")
    for _, r in pub[~pub["passes"]].iterrows():
        print(f"  {r['url']}")
        print(f"    title: {r['title']}")
else:
    print(f"  ✓ All curator-published articles pass the filter")

## Cell 7 — Inspect rejected articles + save audit CSV

Save the rejected list to `data/archive/rejected/<date>_filter-test.csv`. Then
eyeball the top-confidence rejections — those are the most likely false negatives.

In [ ]:
rejected = clf[~clf["passes"]][["url", "title", "source", "article_date",
                                  "top1", "top1_confidence"]].copy()
out_path = Path("data/archive/rejected/2026-05-17_filter-test.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
rejected.to_csv(out_path, index=False)
print(f"Saved {len(rejected)} rejected articles to {out_path}")

print(f"\nTop 15 rejected sorted by classifier top1_confidence")
print(f"(high confidence in rejected = potential false negative — review):")
hc = rejected.nlargest(15, "top1_confidence")
for _, r in hc.iterrows():
    print(f"  conf={r['top1_confidence']:.2f}  [{r['source']}]  {str(r['title'])[:110]}")

## Cell 8 — How to update production

After hand-curating `KEYWORDS` above:

1. Copy the final `KEYWORDS` list into `src/scraping/relevance.py` as `DEFAULT_EDUCATION_KEYWORDS`
2. In `src/scraping/sources.yml`, add `apply_relevance_filter: true` to the `params` block of any broad source:
   ```yaml
   - name: belfast_telegraph
     type: rss
     scraper: src.scraping.rss_adapter
     params:
       feed_url: ...
       apply_relevance_filter: true
   ```
3. Re-run the scraper. Rejected articles appear in `data/archive/rejected/<date>_<source>.csv` per source per run.

Narrow sources (`schoolsweek`, `ofsted`, `epi`, `eef`, `nfer`, etc.) don't need
this filter — their feeds are pure education already (see cell 5 pass rates).